# 🎯 09 — Fine-tune SIM→REAL (Fase 3): fura o teto do F2 com poucas telas reais

> **Célula única — `Runtime → Run all`.** Use **GPU** (Colab). Secret `HF_TOKEN` (baixa o F2 e publica o ajustado).

## Por que
O F2 (notebook 08) já generaliza a estilos **sintéticos** nunca vistos (mAP50 ~99%). Mas telas **reais** de clientes têm detalhes que o sintético não cobre: anti-aliasing, sombras, compressão, fontes proprietárias, ícones. A alavanca comprovada pra fechar esse *gap sim→real* é um **fine-tune curto com <200 telas reais rotuladas**, partindo dos pesos do F2 — não do zero.

## As 4 regras que fazem o fine-tune NÃO piorar o modelo
1. **Parte do `best.pt` do F2**, não do zero — preserva tudo que ele já aprendeu.
2. **Congela o backbone** (`freeze=10`): só a cabeça de detecção se adapta ao domínio real (poucos dados não destroem as features).
3. **Mistura sintético + real** no treino: senão o modelo *esquece* as 52 cartas que suas poucas telas reais não cobrem todas (esquecimento catastrófico).
4. **Valida em REAL held-out separado POR FONTE**: telas do mesmo cliente/sessão nunca aparecem em treino **e** validação ao mesmo tempo — senão o número mente (vazamento de telas quase-iguais).

## Como rotular telas reais barato (human-in-the-loop)
Rotular 200 telas na mão é caro. O truque: o **F2 pré-rotula** (a função `prelabel` roda o modelo nas suas telas e escreve os `.txt` YOLO), você só **corrige** os erros no [LabelImg](https://github.com/HumanSignal/labelImg)/[Roboflow]/CVAT e move pra `real/labels/`. ~10× mais rápido.

## Como usar
- **Com telas reais:** coloque `real/images/*.png` e `real/labels/*.txt` (YOLO 54-classes, nomeados por fonte: `clienteA_001.png`, `clienteB_002.png`...) e rode. Publica `table_yolo11n_ft.onnx` **só se melhorar** o REAL held-out.
- **Sem telas reais:** roda um **stress-test de domínio** (domínio sintético mais duro: cast de cor + blur + compressão) que **prova a maquinaria e mede o ganho** — deixando o caminho real pronto.

## Autossuficiente (checkpoint base)
O passo 0 tenta **baixar** o `table_yolo11n.pt` do F2 do HF. Se ele não estiver publicado (o notebook 08 só passou a publicar o `.pt` recentemente), o F3 **treina um base sintético na hora** a partir do `yolo11n.pt` e parte dele — então `Run all` funciona sozinho. *Dica: re-rodar o 08 (atual) publica o `.pt` e deixa o 09 ~2× mais rápido.*

## Depois
Baixe o `table_yolo11n_ft.onnx`, renomeie pra `poker_vision.onnx` e troque em `backend/models/` — o `/copilot/from-image` passa a usar o detector **ajustado ao real**.

*Classes idênticas às do 08 (52 cartas + `seat` + `button`), então o decode do backend e a geometria de posição continuam iguais.*


In [ ]:
# ============================================================
# FINE-TUNE SIM->REAL (Fase 3) - fura o teto do F2 com POUCAS telas REAIS
# Celula UNICA: Run all. GPU (Colab). Secrets: HF_TOKEN (baixa o F2 e publica o ajustado).
#
# POR QUE: o F2 (notebook 08) ja generaliza a estilos SINTETICOS nunca vistos (mAP50 ~99%),
# mas telas REAIS de clientes tem detalhes que o sintetico nao cobre (anti-aliasing, sombras,
# compressao, fontes proprietarias). A alavanca comprovada pra fechar o gap sim->real e um
# FINE-TUNE curto com <200 telas reais ROTULADAS, partindo dos pesos do F2.
#
# RIGOR (as 4 regras que fazem o fine-tune nao piorar o modelo):
#  1) parte do best.pt do F2 (nao do zero);  2) CONGELA o backbone (so ajusta a cabeca);
#  3) MISTURA sintetico + real (senao esquece as 52 cartas que o real nao cobre todas);
#  4) VALIDA em REAL held-out separado POR FONTE (sem vazamento de telas quase-iguais).
# Sem telas reais? roda um STRESS-TEST de dominio (dominio sintetico mais duro) que PROVA
# a maquinaria e mede o ganho — deixando o caminho real pronto em REAL_DIR.
# ============================================================
import glob
import math
import os
import random
import shutil

get_ipython().system('pip -q install ultralytics huggingface_hub pillow numpy')

import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont


def box(t, ls=()):
    w = max([len(t)] + [len(x) for x in ls]) if ls else len(t)
    print("\n  +" + "-" * (w + 2) + "+")
    print("  | " + t.ljust(w) + " |")
    print("  +" + "-" * (w + 2) + "+")
    for x in ls:
        print("  | " + x.ljust(w) + " |")
    if ls:
        print("  +" + "-" * (w + 2) + "+")


# ---------- CONFIG ----------
ROOT = "/content/ft"
HF_MODEL = "felipesp1983/poker-arena-vision"
BASE_PT_NAME = "table_yolo11n.pt"    # checkpoint do F2 (o notebook 08 publica isso)
REAL_DIR = "/content/real"           # SUAS telas: real/images/*.png (+ real/labels/*.txt YOLO)
EPOCHS_FT, IMGSZ = 40, 640
FREEZE = 10                          # congela o backbone (so a cabeca aprende o dominio real)
LR0_FT = 0.001                       # 1/10 do treino do zero (nao destroi o que o F2 aprendeu)
SYNTH_MIX = 1200                     # sinteticas no treino (mantem as 54 classes vivas)
RANKS, SUITS = "23456789TJQKA", "shdc"
CLASSES = [r + s for r in RANKS for s in SUITS]
SEAT, BUTTON = 52, 53
NAMES = CLASSES + ["seat", "button"]
NC = len(NAMES)
RANK_DISP = {"T": "10"}

box("FINE-TUNE SIM->REAL (F3) - o plano", [
    "0. obter o checkpoint do F2 (best.pt): baixa do HF, OU treina um base agora",
    "1. PRE-ROTULAR suas telas reais com o F2 (voce so CORRIGE - rotulo barato)",
    "2. montar dataset: real (train) + sintetico (mistura) ; REAL held-out por FONTE",
    "3. fine-tune: backbone CONGELADO, LR baixo, poucas epocas",
    "4. medir BASE vs AJUSTADO no REAL held-out -> so publica se MELHOROU",
])

get_ipython().system('apt-get -qq install -y fonts-liberation fonts-dejavu-core fonts-freefont-ttf >/dev/null 2>&1')
FONTS = glob.glob("/usr/share/fonts/truetype/**/*.ttf", recursive=True)
FONTS = [f for f in FONTS if not any(b in f.lower() for b in ("emoji", "symbol", "mono"))][:14] or [None]


# ---------- gerador sintetico (mesma ideia do 08: cartas+jogadores+botao) ----------
def draw_suit(d, suit, cx, cy, s, color):
    if suit == "d":
        d.polygon([(cx, cy - s), (cx + s * .7, cy), (cx, cy + s), (cx - s * .7, cy)], fill=color)
    elif suit == "h":
        d.pieslice([cx - s, cy - s, cx, cy], 0, 360, fill=color)
        d.pieslice([cx, cy - s, cx + s, cy], 0, 360, fill=color)
        d.polygon([(cx - s * .98, cy - s * .1), (cx + s * .98, cy - s * .1), (cx, cy + s)], fill=color)
    elif suit == "s":
        d.polygon([(cx, cy - s), (cx + s * .85, cy + s * .4), (cx - s * .85, cy + s * .4)], fill=color)
        d.pieslice([cx - s * .85, cy - s * .1, cx, cy + s * .6], 0, 360, fill=color)
        d.pieslice([cx, cy - s * .1, cx + s * .85, cy + s * .6], 0, 360, fill=color)
        d.rectangle([cx - s * .14, cy + s * .2, cx + s * .14, cy + s], fill=color)
    else:
        r = s * .5
        d.ellipse([cx - r, cy - s * .9, cx + r, cy + r * .2], fill=color)
        d.ellipse([cx - s * .9, cy - r * .3, cx - s * .9 + 2 * r, cy - r * .3 + 2 * r], fill=color)
        d.ellipse([cx + s * .9 - 2 * r, cy - r * .3, cx + s * .9, cy - r * .3 + 2 * r], fill=color)
        d.rectangle([cx - s * .14, cy, cx + s * .14, cy + s], fill=color)


def render_card(rng, rank, suit, w):
    h = int(w * 1.4)
    img = Image.new("RGB", (w, h), (0, 0, 0))
    bg = tuple(rng.randint(238, 255) for _ in range(3))
    d = ImageDraw.Draw(img)
    d.rounded_rectangle([0, 0, w - 1, h - 1], radius=rng.randint(3, max(4, w // 8)), fill=bg,
                        outline=tuple(rng.randint(150, 200) for _ in range(3)), width=rng.randint(1, 2))
    four = rng.random() < 0.4
    cmap = {"s": (25, 25, 25), "c": ((20, 120, 40) if four else (25, 25, 25)),
            "h": (200, 30, 40), "d": ((30, 90, 210) if four else (200, 30, 40))}
    col = tuple(int(np.clip(c + rng.randint(-25, 25), 0, 255)) for c in cmap[suit])
    try:
        f = ImageFont.truetype(rng.choice(FONTS), int(w * rng.uniform(0.30, 0.40)))
    except Exception:
        f = ImageFont.load_default()
    d.text((w * 0.08, h * 0.03), RANK_DISP.get(rank, rank), font=f, fill=col)
    draw_suit(d, suit, int(w * 0.24), int(h * 0.30), w * 0.11, col)
    draw_suit(d, suit, w // 2, int(h * 0.62), w * rng.uniform(0.22, 0.28), col)
    return img


def gen_sample(rng, sseed, hard=False):
    """Mesa 2D + labels YOLO (54 classes). hard=True = dominio 'mais real/dificil'
    (compressao, cast de cor, leve blur) usado no stress-test quando falta tela real."""
    W, H = 900, 600
    srng = random.Random(sseed)
    felt = tuple(srng.randint(15, 110) for _ in range(3))
    img = Image.new("RGB", (W, H), felt)
    d = ImageDraw.Draw(img)
    if srng.random() < 0.7:
        d.ellipse([W * .08, H * .03, W * .92, H * .97],
                  fill=tuple(min(255, c + srng.randint(8, 30)) for c in felt))
    labels = []
    n = rng.randint(2, 9)
    cx0, cy0, rx, ry = W / 2, H / 2, W * 0.40, H * 0.44
    seats = [(cx0 + rx * math.cos(math.radians(90 + i * 360 / n)),
              cy0 + ry * math.sin(math.radians(90 + i * 360 / n))) for i in range(n)]
    for (sx, sy) in seats:
        rr = srng.randint(20, 34)
        acol = tuple(srng.randint(20, 235) for _ in range(3))
        d.ellipse([sx - rr, sy - rr, sx + rr, sy + rr], fill=acol, outline=(235, 235, 235), width=2)
        labels.append((SEAT, sx / W, sy / H, (2 * rr) / W, (2 * rr) / H))
    deck = CLASSES[:]
    rng.shuffle(deck)
    nb = rng.choice([0, 3, 3, 4, 5])
    hole, board = deck[:2], deck[2:2 + nb]
    cw = int(srng.uniform(40, 84))

    def place(card, cx, cy):
        ci = render_card(rng, card[0], card[1], cw)
        img.paste(ci, (int(cx - ci.width / 2), int(cy - ci.height / 2)))
        labels.append((CLASSES.index(card), cx / W, cy / H, ci.width / W, ci.height / H))
    if board:
        gap = int(cw * 1.3)
        x0 = W // 2 - (len(board) - 1) * gap // 2
        by = H // 2 + rng.randint(-25, 10)
        for i, c in enumerate(board):
            place(c, x0 + i * gap + rng.randint(-4, 4), by)
    hx, hy = W // 2 + rng.randint(-50, 50), int(H * 0.76) + rng.randint(-12, 10)
    for i, c in enumerate(hole):
        place(c, hx - 46 + i * 92, hy)
    bseat = rng.randrange(n)
    sxb, syb = seats[bseat]
    ux, uy = (cx0 - sxb), (cy0 - syb)
    lg = math.hypot(ux, uy) or 1.0
    ux, uy = ux / lg, uy / lg
    bx, by2 = sxb + ux * 16 - uy * 42, syb + uy * 16 + ux * 42
    brr = srng.randint(10, 16)
    d.ellipse([bx - brr, by2 - brr, bx + brr, by2 + brr], fill=(232, 200, 60), outline=(120, 95, 10), width=2)
    labels.append((BUTTON, bx / W, by2 / H, (2 * brr) / W, (2 * brr) / H))
    if hard:  # 'mais real': cast de cor + blur + compressao forte
        img = Image.blend(img, Image.new("RGB", (W, H), (srng.randint(0, 60), srng.randint(0, 40), srng.randint(0, 60))), 0.12)
        img = img.filter(ImageFilter.GaussianBlur(rng.uniform(0.4, 1.3)))
    elif rng.random() < 0.5:
        img = img.filter(ImageFilter.GaussianBlur(rng.uniform(0, 0.8)))
    return img, labels


def write_split(split, imgs_labels):
    os.makedirs(f"{ROOT}/images/{split}", exist_ok=True)
    os.makedirs(f"{ROOT}/labels/{split}", exist_ok=True)
    for k, (img, labels) in enumerate(imgs_labels):
        q = random.Random(k).randint(35, 70)  # compressao variavel (mais 'real')
        img.save(f"{ROOT}/images/{split}/{split}_{k:05}.jpg", quality=q)
        with open(f"{ROOT}/labels/{split}/{split}_{k:05}.txt", "w") as fh:
            fh.write("\n".join(f"{c} {x:.5f} {y:.5f} {w:.5f} {h:.5f}" for c, x, y, w, h in labels))


# ---------- STEP 0: baixar o checkpoint base do F2 ----------
box("[0/4] Obtendo o checkpoint base do F2 (best.pt)")
from huggingface_hub import hf_hub_download
from ultralytics import YOLO
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")


def train_synth_base():
    """Fallback: se o .pt do F2 nao esta no HF (o 08 so publicou o .onnx), treina um
    BASE sintetico agora a partir do yolo11n.pt — F3 fica autossuficiente. E um STOPGAP:
    o base ideal e o F2 do 08 (98.7%); re-rode o 08 (atual, publica o .pt) e o 09 parte
    dele. Aqui: 4000 imgs / 80 epocas pra o base convergir bem (nao subtreinar)."""
    broot = f"{ROOT}/base"
    rngb = random.Random(1)
    for split, n, ids in (("train", 4000, range(0, 80)), ("val", 500, range(80, 100))):
        os.makedirs(f"{broot}/images/{split}", exist_ok=True)
        os.makedirs(f"{broot}/labels/{split}", exist_ok=True)
        for k in range(n):
            img, labels = gen_sample(rngb, rngb.choice(list(ids)))
            img.save(f"{broot}/images/{split}/{k:05}.jpg", quality=88)
            with open(f"{broot}/labels/{split}/{k:05}.txt", "w") as fh:
                fh.write("\n".join(f"{c} {x:.5f} {y:.5f} {w:.5f} {h:.5f}" for c, x, y, w, h in labels))
    with open(f"{broot}/data.yaml", "w") as fh:
        fh.write(f"path: {broot}\ntrain: images/train\nval: images/val\nnc: {NC}\nnames: {NAMES}\n")
    b = YOLO("yolo11n.pt")
    b.train(data=f"{broot}/data.yaml", epochs=80, imgsz=IMGSZ, batch=32,
            patience=20, device=0, project=ROOT, name="base", verbose=True)
    return str(b.trainer.best)


try:  # rapido quando o 08 (atual) ja publicou o .pt
    base_pt = hf_hub_download(HF_MODEL, BASE_PT_NAME, token=HF_TOKEN, local_dir=ROOT)
    print("checkpoint base (do HF):", base_pt)
except Exception as e:
    print(f"(.pt do F2 nao esta no HF: {type(e).__name__} -> treino um base sintetico agora)")
    print(" dica: re-rodar o notebook 08 (atual) publica o .pt e deixa o 09 ~2x mais rapido")
    base_pt = train_synth_base()
    print("checkpoint base (sintetico, treinado agora):", base_pt)
base = YOLO(base_pt)

# ---------- STEP 1: telas REAIS (ou stress-test de dominio) ----------
real_imgs = sorted(glob.glob(f"{REAL_DIR}/images/*.*"))
have_real = len(real_imgs) >= 20 and len(glob.glob(f"{REAL_DIR}/labels/*.txt")) >= 20
if have_real:
    box("[1/4] Telas REAIS encontradas", [f"{len(real_imgs)} imagens + labels em {REAL_DIR}"])
    # split POR FONTE: usa o prefixo do nome do arquivo (ex.: 'pokerX_001.png' -> fonte 'pokerX')
    def source(p):
        return os.path.basename(p).split("_")[0]
    srcs = sorted({source(p) for p in real_imgs})
    val_srcs = set(srcs[::4]) or {srcs[-1]}  # ~25% das FONTES viram held-out (sem vazamento)
    tr = [p for p in real_imgs if source(p) not in val_srcs]
    va = [p for p in real_imgs if source(p) in val_srcs]
    print(f"fontes: {len(srcs)} | treino {len(tr)} imgs | held-out {len(va)} imgs (fontes {sorted(val_srcs)})")
    for split, ps in (("train", tr), ("val", va)):
        os.makedirs(f"{ROOT}/images/{split}", exist_ok=True)
        os.makedirs(f"{ROOT}/labels/{split}", exist_ok=True)
        for p in ps:
            shutil.copy(p, f"{ROOT}/images/{split}/{os.path.basename(p)}")
            lp = f"{REAL_DIR}/labels/{os.path.splitext(os.path.basename(p))[0]}.txt"
            if os.path.exists(lp):
                shutil.copy(lp, f"{ROOT}/labels/{split}/{os.path.splitext(os.path.basename(p))[0]}.txt")
    DEMO = False
else:
    box("[1/4] Sem telas reais -> STRESS-TEST de dominio", [
        "gero um dominio sintetico MAIS DURO (cast/blur/compressao) como 'pseudo-real'",
        "prova a maquinaria e MEDE o ganho; troque por telas reais em " + REAL_DIR,
        "PRE-ROTULO: rode base.predict nas suas telas p/ rotulo barato (veja o codigo)",
    ])
    rng = random.Random(2024)
    write_split("train", [gen_sample(rng, rng.randint(200, 260), hard=True) for _ in range(160)])
    write_split("val", [gen_sample(rng, rng.randint(260, 300), hard=True) for _ in range(80)])
    DEMO = True

# mistura sintetica 'facil' no treino (anti-esquecimento das 54 classes)
box("[2/4] Misturando sintetico ao treino (anti-esquecimento)")
rngm = random.Random(7)
os.makedirs(f"{ROOT}/images/train", exist_ok=True)
os.makedirs(f"{ROOT}/labels/train", exist_ok=True)
for k in range(SYNTH_MIX):
    img, labels = gen_sample(rngm, rngm.randint(0, 79))
    img.save(f"{ROOT}/images/train/synth_{k:05}.jpg", quality=88)
    with open(f"{ROOT}/labels/train/synth_{k:05}.txt", "w") as fh:
        fh.write("\n".join(f"{c} {x:.5f} {y:.5f} {w:.5f} {h:.5f}" for c, x, y, w, h in labels))
with open(f"{ROOT}/data.yaml", "w") as fh:
    fh.write(f"path: {ROOT}\ntrain: images/train\nval: images/val\nnc: {NC}\nnames: {NAMES}\n")

# como PRE-ROTULAR telas reais de graca (rode isto, corrija no LabelImg/Roboflow, mova pra labels/):
def prelabel(model, img_dir, out_dir):
    """Roda o F2 nas telas e escreve labels YOLO (voce so CORRIGE). Rotulo ~10x mais barato."""
    os.makedirs(out_dir, exist_ok=True)
    for p in sorted(glob.glob(f"{img_dir}/*.*")):
        r = model.predict(p, imgsz=IMGSZ, conf=0.35, verbose=False)[0]
        w, h = Image.open(p).size
        with open(f"{out_dir}/{os.path.splitext(os.path.basename(p))[0]}.txt", "w") as fh:
            for b in r.boxes:
                cx, cy, bw, bh = b.xywhn[0].tolist()
                fh.write(f"{int(b.cls[0])} {cx:.5f} {cy:.5f} {bw:.5f} {bh:.5f}\n")

# ---------- STEP 3: fine-tune (backbone congelado, LR baixo) ----------
box("[3/4] Fine-tune (backbone CONGELADO, LR baixo, poucas epocas)")
ft = YOLO(base_pt)
ft.train(data=f"{ROOT}/data.yaml", epochs=EPOCHS_FT, imgsz=IMGSZ, batch=32, freeze=FREEZE,
         lr0=LR0_FT, patience=10, device=0, project=ROOT, name="ft", verbose=True)

# ---------- STEP 4: BASE vs AJUSTADO no held-out (real) + export/publish se melhorou ----------
box("[4/4] BASE vs AJUSTADO no held-out + export")
m_base = base.val(data=f"{ROOT}/data.yaml", split="val", verbose=False)
m_ft = ft.val(data=f"{ROOT}/data.yaml", split="val", verbose=False)
b50, f50 = float(m_base.box.map50), float(m_ft.box.map50)
ganho = (f50 - b50) * 100
onnx = ft.export(format="onnx", imgsz=IMGSZ, opset=12)
box("RESULTADO (fine-tune sim->real)", [
    f"held-out {'DEMO (dominio duro)' if DEMO else 'REAL'} - mAP50 BASE:    {b50 * 100:.1f}%",
    f"held-out {'DEMO (dominio duro)' if DEMO else 'REAL'} - mAP50 AJUSTADO: {f50 * 100:.1f}%",
    f"GANHO do fine-tune: {ganho:+.1f} pontos",
    ("stress-test: prova a maquinaria; troque por telas reais p/ o ganho de verdade"
     if DEMO else "ganho medido em telas REAIS held-out (sem vazamento, split por fonte)"),
])
print("ONNX ajustado:", onnx)

publica = (not DEMO) and (f50 >= b50)  # so publica se ajustou em REAL e NAO piorou
if publica:
    try:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        api.upload_file(path_or_fileobj=str(onnx), path_in_repo="table_yolo11n_ft.onnx",
                        repo_id=HF_MODEL, repo_type="model")
        print("publicado: table_yolo11n_ft.onnx (troque no backend por poker_vision.onnx)")
    except Exception as e:
        print("(HF pulado:", e, ")")
else:
    print("NAO publicado (DEMO, ou o fine-tune nao melhorou o REAL) - ajuste dados/hparams.")
print("\nFIM. Com telas reais rotuladas em", REAL_DIR, "isto fecha o gap sim->real do detector.")